# Neural Flow: Conditioning a Learned Vector Field

The other notebooks build the prior as a differentiable map from Gaussian increments to a field (`GaussianIncrementFlow`), but for many targets no such map exists and all we have are unconditional samples. Here we learn the prior instead: a time-conditioned U-Net velocity field $v_\theta(x, t)$ is fitted by flow matching to samples of a **Brown–Resnick** max-stable field. Once wrapped in a `NeuralFlow`, the same `EqualityCondition` + `GuidanceDPS` posterior sampling works unchanged, which we show on an areal-disaggregation problem. A GPU is strongly recommended for training.

In [ ]:
import math
import sys
from pathlib import Path

import gpytorch
import torch
import torch.nn as nn

from flow_inference.conditions import EqualityCondition
from flow_inference.flow_utils import build_grid_in_box
from flow_inference.flows import NeuralFlow
from flow_inference.guidance import GuidanceDPS
from flow_inference.maps import BrownResnickMap
from flow_inference.sampler import IntegratorSampler
from flow_inference.training_samplers import TrainingSampler

sys.path.insert(0, str(Path.cwd().parent))  # examples/, for plotting.py
from plotting import shared_range, show_fields, show_overlay

torch.set_default_dtype(torch.float32)  # the learned flow does not need float64, and float32 trains faster

In [ ]:
SEED = 0
torch.manual_seed(SEED)

H, W = 32, 32               # grid size
num_samples = 5             # samples drawn from the prior / posterior
num_training_samples = 10_000  # size of the pregenerated training pool
num_train_steps = 10_000    # flow-matching optimisation steps
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Training data

Build the Brown–Resnick map and wrap it in a `TrainingSampler`, which pregenerates a pool of unconditional samples. These samples are all the neural flow ever sees of the target.

In [ ]:
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.MaternKernel(nu=1.5)).to(device)
kernel.outputscale = 1.0
kernel.base_kernel.lengthscale = 0.5
brown_resnick_map = BrownResnickMap(H=H, W=W, kernel=kernel, num_storms=100, device=device)

training_sampler = TrainingSampler(num_training_samples, brown_resnick_map)

real_samples = training_sampler(num_samples).to(device)
show_fields(real_samples.view(real_samples.shape[0], H, W))

## 2. The vector field

$v_\theta(x, t)$ is a small U-Net acting on the field as a $1 \times H \times W$ image, with Fourier time embeddings injected at every resolution by FiLM (per-channel scale and shift). Multiplicative time conditioning lets the network rescale the velocity across noise levels, which helps with heavy-tailed targets.

In [ ]:
class TimeEmbedding(nn.Module):
    """Fourier-feature embedding of t in [0, 1], followed by an MLP."""

    def __init__(self, dim: int):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.dim = dim

    def forward(self, t):
        freqs = torch.arange(1, self.dim // 2 + 1, device=t.device, dtype=t.dtype)
        angles = 2 * math.pi * t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(emb)


class ConvBlock(nn.Module):
    """Two 3x3 convs with GroupNorm + SiLU and FiLM time conditioning in between."""

    def __init__(self, in_ch, out_ch, time_dim, gn_groups=8):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(gn_groups, out_ch)
        self.t_proj = nn.Linear(time_dim, out_ch * 2)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(gn_groups, out_ch)
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        scale, shift = self.t_proj(t_emb)[:, :, None, None].chunk(2, dim=1)
        h = h * (1.0 + scale) + shift   # identity at scale = shift = 0
        h = self.act(self.norm2(self.conv2(h)))
        return h


class VectorFieldUNet(nn.Module):
    """Time-conditioned U-Net v_theta(x, t) with concat skips; `depth` halvings of the grid."""

    def __init__(self, in_ch=1, out_ch=1, base_ch=32, time_dim=64, depth=2, gn_groups=8):
        super().__init__()
        self.depth = depth
        self.time_mlp = TimeEmbedding(time_dim)

        chs = [base_ch * (2 ** i) for i in range(depth + 1)]  # e.g. depth=2 -> [C, 2C, 4C]

        self.enc_blocks = nn.ModuleList()
        self.enc_blocks.append(ConvBlock(in_ch, chs[0], time_dim, gn_groups))
        for i in range(depth):
            self.enc_blocks.append(ConvBlock(chs[i], chs[i + 1], time_dim, gn_groups))
        self.pool = nn.MaxPool2d(2)

        self.ups = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for i in reversed(range(depth)):
            self.ups.append(nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(chs[i + 1], chs[i], 1),
            ))
            self.dec_blocks.append(ConvBlock(chs[i] * 2, chs[i], time_dim, gn_groups))

        self.out = nn.Conv2d(chs[0], out_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)

        skips = []
        h = self.enc_blocks[0](x, t_emb)
        for i in range(self.depth):
            skips.append(h)
            h = self.enc_blocks[i + 1](self.pool(h), t_emb)

        for up, dec, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = dec(torch.cat([skip, up(h)], dim=1), t_emb)

        return self.out(h)


network = VectorFieldUNet(in_ch=1, out_ch=1, base_ch=64, time_dim=64, depth=2).to(device)

## 3. Train the flow

Wrap the network in a `NeuralFlow` and fit it by flow matching, i.e. regressing the probability-flow ODE velocity on samples from the training pool.

In [ ]:
flow = NeuralFlow(H, W, network)
flow.fit(training_sampler, num_steps=num_train_steps, batch_size=64, lr=2e-4)

## 4. Prior samples

Integrate the learned flow with no guidance and check the samples look like the training data above.

In [ ]:
sampler = IntegratorSampler(flow=flow)
samples = sampler.sample(num_samples=num_samples, num_steps=100, solver="heun")
show_fields(samples)

## 5. Areal disaggregation

Observe only the *totals* over 16 irregular regions (think census tracts or satellite footprints; here a Voronoi tessellation of random seed pixels) and recover a plausible fine-scale field. There is no closed-form map to draw a ground truth from, so it is a fresh draw from the training sampler.

In [ ]:
class RegionMassCondition(EqualityCondition):
    """Match the summed field within each of R regions to a target total."""

    def __init__(self, H, W, region_ids, region_targets, sigma=1e-2):
        super().__init__(sigma=sigma)
        self.H, self.W = H, W
        self.R = int(region_targets.numel())
        self.register_buffer("region_ids", region_ids.reshape(-1).long())     # (N,) in {0..R-1}
        self.register_buffer("region_targets", region_targets.reshape(-1))    # (R,)

    def residual(self, x):
        B = x.shape[0]
        x = x.reshape(B, -1)                                                   # (B, N)
        sums = torch.zeros(B, self.R, device=x.device, dtype=x.dtype)
        sums = sums.index_add(1, self.region_ids, x)                          # (B, R)
        return (sums - self.region_targets[None]).unsqueeze(1)                # (B, 1, R)


# Voronoi tessellation from K random seed pixels.
K = 16
coords     = build_grid_in_box(H, W, device=device, dtype=torch.float32)      # (N, 2)
region_ids = torch.cdist(coords, coords[torch.randperm(H * W, device=device)[:K]]).argmin(dim=1)
ones       = torch.ones(H * W, device=device, dtype=torch.get_default_dtype())
counts     = torch.zeros(K, device=device, dtype=ones.dtype).index_add(0, region_ids, ones)

gt             = training_sampler(1).reshape(1, flow.latent_dim).to(device).detach()
gt_flat        = gt.reshape(-1)
region_targets = torch.zeros(K, device=device, dtype=gt_flat.dtype).index_add(0, region_ids, gt_flat)
condition      = RegionMassCondition(H, W, region_ids, region_targets, sigma=1e-5).to(device)

guidance = GuidanceDPS(flow, condition, max_norm=1e1).to(device)
post     = sampler.sample(num_samples, guidance=guidance, num_steps=500, sde=True)

coarse = (region_targets / counts.clamp(min=1))[region_ids].view(1, H, W)     # observed region means
vmin, vmax = shared_range(gt, coarse, post)
plot_kw = dict(h=H, w=W, n=num_samples, boundaries=region_ids, vmin=vmin, vmax=vmax)
print("truth | observed region means (the only data) | posterior samples")
show_overlay(gt,     titles=["ground truth"], **plot_kw)
show_overlay(coarse, titles=["region means"], **plot_kw)
show_overlay(post, **plot_kw)